In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


In [3]:
# (CHANGE THESE ACCORDING TO YOUR DRIVE)
STUDENTLIFE_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/data/raw/studentlife"
#OUTPUT_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/results_realistic_studentlife"

#os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Paths configured.")


Paths configured.


In [4]:
# Loading StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")

stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")


Loading StudentLife...
Stress: (2408, 5)
Activity: (833, 9)
Sleep: (1644, 7)


In [5]:
stress_raw.head()

,null,resp_time,student_id,level,location
0,3,1364121467,Stress_u20,NaN,NaN
1,"43.70413179,-72.28882107",1364121469,Stress_u20,NaN,NaN
2,1,1364121470,Stress_u20,NaN,NaN
3,"43.70413179,-72.28882107",1364121793,Stress_u20,NaN,NaN
4,2,1364121465,Stress_u20,NaN,NaN


In [6]:
activity_raw.head()

,Social2,null,resp_time,student_id,other_relaxing,other_working,relaxing,working,location
0,2,4,1364884639,Activity_u33,NaN,NaN,NaN,NaN,NaN
1,3,2,1364590835,Activity_u33,NaN,NaN,NaN,NaN,NaN
2,2,1,1364504673,Activity_u33,NaN,NaN,NaN,NaN,NaN
3,2,2,1364677565,Activity_u33,NaN,NaN,NaN,NaN,NaN
4,2,2,1364765272,Activity_u33,NaN,NaN,NaN,NaN,NaN


In [7]:
sleep_raw.head()

,hour,location,rate,resp_time,social,student_id,null
0,9,"43.70357146,-72.29017646",1,1364761981,1,Sleep_u22,NaN
1,NaN,NaN,NaN,1364122237,NaN,Sleep_u22,6
2,NaN,NaN,NaN,1364122241,NaN,Sleep_u22,"43.70629505,-72.28825598"
3,NaN,NaN,NaN,1364122243,NaN,Sleep_u22,9
4,NaN,NaN,NaN,1364122235,NaN,Sleep_u22,6


In [8]:
# Cleaning the STRESS DATASET
# =========================
print("\n DATASET: STRESS")
print("Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()

# Dropping the 'null' column
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Converting  timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Cleaning  student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)



 DATASET: STRESS
Shows self-reported student stress levels over time

Cleaning StudentLife Stress...

Initial Stress Dataset:
       resp_time  student_id level                  location
0     1364121467  Stress_u20   NaN                       NaN
1     1364121469  Stress_u20   NaN                       NaN
2     1364121470  Stress_u20   NaN                       NaN
3     1364121793  Stress_u20   NaN                       NaN
4     1364121465  Stress_u20   NaN                       NaN
...          ...         ...   ...                       ...
2403  1366085234  Stress_u41     1  43.70428777,-72.28778708
2404  1367003837  Stress_u41     1  43.70543645,-72.28828558
2405  1367193608  Stress_u41     2  43.70535262,-72.28819131
2406  1367312977  Stress_u41     3  43.70526777,-72.28881679
2407  1368000327  Stress_u41     3                   Unknown

[2408 rows x 4 columns]

After converting resp_time to timestamp:
       resp_time  student_id level                  location  \
0     1364

In [9]:
#renaming the level column
stress_clean = stress_clean.rename(columns={'level': 'stress_level'})
print("\nAfter renaming level → stress_level:")
print(stress_clean)

#Converting stress_level to numeric data
stress_clean['stress_level'] = pd.to_numeric(stress_clean['stress_level'], errors='coerce')
print("\nAfter converting stress_level to numeric:")
print(stress_clean)

#dropping missing stress values
stress_clean = stress_clean.dropna(subset=['stress_level'])
print("\nAfter dropping NaN stress levels:")
print(stress_clean)

#Explicit float conversion
stress_clean['stress_level'] = stress_clean['stress_level'].astype(float)
print("\nFinal cleaned Stress dataset:")
print(stress_clean)



After renaming level → stress_level:
       resp_time student_id stress_level                  location  \
0     1364121467        u20          NaN                       NaN   
1     1364121469        u20          NaN                       NaN   
2     1364121470        u20          NaN                       NaN   
3     1364121793        u20          NaN                       NaN   
4     1364121465        u20          NaN                       NaN   
...          ...        ...          ...                       ...   
2403  1366085234        u41            1  43.70428777,-72.28778708   
2404  1367003837        u41            1  43.70543645,-72.28828558   
2405  1367193608        u41            2  43.70535262,-72.28819131   
2406  1367312977        u41            3  43.70526777,-72.28881679   
2407  1368000327        u41            3                   Unknown   

               timestamp  
0    2013-03-24 10:37:47  
1    2013-03-24 10:37:49  
2    2013-03-24 10:37:50  
3    2013-03-

In [10]:
# Code for Activity data cleaning
print("\n DATASET: ACTIVITY")
print(" Shows students' daily activity levels (social, working, relaxing)")

print("\nCleaning StudentLife Activity")

activity_clean = activity_raw.copy()

# Drop the 'null' coloumns
if 'null' in activity_clean.columns:
    activity_clean = activity_clean.drop(columns=['null'])

print("\nInitial Activity Dataset:")
print(activity_clean)

# Convert timestamp

activity_clean['timestamp'] = pd.to_datetime(activity_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(activity_clean)

# Clean student_id
activity_clean['student_id'] = activity_clean['student_id'].str.replace('Activity_', '')
print("\nAfter cleaning student_id:")
print(activity_clean)



 DATASET: ACTIVITY
 Shows students' daily activity levels (social, working, relaxing)

Cleaning StudentLife Activity

Initial Activity Dataset:
    Social2   resp_time    student_id other_relaxing other_working relaxing  \
0         2  1364884639  Activity_u33            NaN           NaN      NaN   
1         3  1364590835  Activity_u33            NaN           NaN      NaN   
2         2  1364504673  Activity_u33            NaN           NaN      NaN   
3         2  1364677565  Activity_u33            NaN           NaN      NaN   
4         2  1364765272  Activity_u33            NaN           NaN      NaN   
..      ...         ...           ...            ...           ...      ...   
828     NaN  1365660037  Activity_u23              1             1        1   
829     NaN  1365832850  Activity_u23              1             2        1   
830     NaN  1365919216  Activity_u23              1             1        3   
831     NaN  1366823935  Activity_u23              1             

In [11]:


# key columns of studentlife dataset

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
    if col not in activity_clean.columns:
        activity_clean[col] = np.nan



# Converting to numeric

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
  activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")



# Computing interpretable scores

activity_clean["workload_score"] = activity_clean[["working", "other_working"]].sum(axis=1, min_count=1)
activity_clean["recovery_score"] = activity_clean[["relaxing", "other_relaxing"]].sum(axis=1, min_count=1)
activity_clean["social_score"] = activity_clean["Social2"]


print("\nAfter computing workload_score / recovery_score / social_score:")

print(activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]].head())



# Keep relevant columns and drop rows where ALL scores are missing

activity_clean = activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]]
activity_clean = activity_clean.dropna( subset=["workload_score", "recovery_score", "social_score"], how="all" ).copy()



print("\nFinal cleaned Activity dataset (3 scores):")
print(activity_clean.head())




After computing workload_score / recovery_score / social_score:
  student_id           timestamp  workload_score  recovery_score  social_score
0        u33 2013-04-02 06:37:19             NaN             NaN           2.0
1        u33 2013-03-29 21:00:35             NaN             NaN           3.0
2        u33 2013-03-28 21:04:33             NaN             NaN           2.0
3        u33 2013-03-30 21:06:05             NaN             NaN           2.0
4        u33 2013-03-31 21:27:52             NaN             NaN           2.0

Final cleaned Activity dataset (3 scores):
  student_id           timestamp  workload_score  recovery_score  social_score
0        u33 2013-04-02 06:37:19             NaN             NaN           2.0
1        u33 2013-03-29 21:00:35             NaN             NaN           3.0
2        u33 2013-03-28 21:04:33             NaN             NaN           2.0
3        u33 2013-03-30 21:06:05             NaN             NaN           2.0
4        u33 2013-03-3

In [12]:
# Sleep dataset processing

# Print dataset name
print("\ DATASET: SLEEP")

# Describe dataset purpose
print("Shows students' self-reported sleep duration in hours")

# Start cleaning process
print("\nCleaning StudentLife Sleep...")

# Create a copy of raw dataset
sleep_clean = sleep_raw.copy()

# Remove irrelevant null column if present
if 'null' in sleep_clean.columns:
    sleep_clean = sleep_clean.drop(columns=['null'])

# Display initial dataset
print("\nInitial Sleep Dataset:")
print(sleep_clean)

# Convert response time to timestamp
sleep_clean['timestamp'] = pd.to_datetime(sleep_clean['resp_time'], unit='s')

# Show dataset after timestamp conversion
print("\nAfter converting resp_time to timestamp:")
print(sleep_clean)

\ DATASET: SLEEP
Shows students' self-reported sleep duration in hours

Cleaning StudentLife Sleep...

Initial Sleep Dataset:
     hour                  location rate   resp_time social student_id
0       9  43.70357146,-72.29017646    1  1364761981      1  Sleep_u22
1     NaN                       NaN  NaN  1364122237    NaN  Sleep_u22
2     NaN                       NaN  NaN  1364122241    NaN  Sleep_u22
3     NaN                       NaN  NaN  1364122243    NaN  Sleep_u22
4     NaN                       NaN  NaN  1364122235    NaN  Sleep_u22
...   ...                       ...  ...         ...    ...        ...
1639    6  43.70684315,-72.28727941    2  1365967467      1  Sleep_u54
1640    7  43.70684315,-72.28727941    1  1366046298      1  Sleep_u54
1641    6  43.70684315,-72.28727941    1  1366132595      1  Sleep_u54
1642    4  43.70611494,-72.28821312    3  1366222570      1  Sleep_u54
1643    6                   Unknown    2  1366574628      1  Sleep_u54

[1644 rows x 6 column

In [13]:
# Clean student_id by removing prefix
sleep_clean['student_id'] = sleep_clean['student_id'].str.replace('Sleep_', '')

# Show dataset after cleaning student_id
print("\nAfter cleaning student_id:")
print(sleep_clean)

# Convert hour column to numeric sleep_hours
sleep_clean['sleep_hours'] = pd.to_numeric(sleep_clean['hour'], errors='coerce')

# Show dataset after converting sleep hours
print("\nAfter converting hour to sleep_hours:")
print(sleep_clean)

# Keep only relevant columns and remove missing values
sleep_clean = sleep_clean[['student_id', 'timestamp', 'sleep_hours']].dropna()

# Display final cleaned dataset
print("\nFinal cleaned Sleep dataset:")
print(sleep_clean)


After cleaning student_id:
     hour                  location rate   resp_time social student_id  \
0       9  43.70357146,-72.29017646    1  1364761981      1        u22   
1     NaN                       NaN  NaN  1364122237    NaN        u22   
2     NaN                       NaN  NaN  1364122241    NaN        u22   
3     NaN                       NaN  NaN  1364122243    NaN        u22   
4     NaN                       NaN  NaN  1364122235    NaN        u22   
...   ...                       ...  ...         ...    ...        ...   
1639    6  43.70684315,-72.28727941    2  1365967467      1        u54   
1640    7  43.70684315,-72.28727941    1  1366046298      1        u54   
1641    6  43.70684315,-72.28727941    1  1366132595      1        u54   
1642    4  43.70611494,-72.28821312    3  1366222570      1        u54   
1643    6                   Unknown    2  1366574628      1        u54   

               timestamp  
0    2013-03-31 20:33:01  
1    2013-03-24 10:50:37  
2 

In [14]:
# Sahitya_week2 added code for feature engineering for stress dataset
print("CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)")

for df, time_col in [  # loop through each dataframe and its timestamp column name

    (stress_clean, "timestamp"),  # pair: stress dataframe + timestamp column

    (activity_clean, "timestamp"),  # pair: activity dataframe + timestamp column

    (sleep_clean, "timestamp"),  # pair: sleep dataframe + timestamp column

]:

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")  # convert to datetime;

sleep_clean.head()

CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)


,student_id,timestamp,sleep_hours
0,u22,2013-03-31 20:33:01,9.0
6,u22,2013-04-03 17:19:03,7.0
7,u22,2013-03-27 07:35:58,6.0
8,u22,2013-03-28 19:30:33,8.0
9,u22,2013-03-28 03:06:53,6.0


the following code is written by jaimil kothari - week 2

In [15]:


def add_student_zscore(df, id_col, value_col, new_col, time_col=None):  # to define helper function to add a z-score column

    if time_col is None:  # checking if a time column name was provided or not
        # falling back to index order if timestamp isn't provided
        time_col = None  # keeping it none explicitly

    if time_col is not None and time_col in df.columns:  # if time_col exists then doing sort by id + time
        df = df.sort_values([id_col, time_col]).copy()  # sorting data to ensure expanding stats follow time order

    else:  # else sort only by student id
        df = df.sort_values([id_col]).copy()  # sorting by student id for deterministic order

    def _exp_z(s: pd.Series) -> pd.Series:  # this is an inner function to compute expanding z-score for one student series
        exp_mean = s.expanding(min_periods=1).mean()  # expanding mean up to each point
        exp_std = s.expanding(min_periods=2).std().fillna(0.0)  # expanding std; needs >=2 points
        exp_std = exp_std.replace(0, np.nan).fillna(1.0)  # replacing 0 std with 1 to avoid divide-by-zero
        return (s - exp_mean) / exp_std  # computing z-score using expanding mean/std


    df[new_col] = df.groupby(id_col)[value_col].transform(_exp_z)  # expanding z-score per student
    return df  # returning dataframe with new z-score column added


stress_clean = add_student_zscore(stress_clean, "student_id", "stress_level", "stress_z", time_col="timestamp")  # adding stress_z

for col, zcol in [("workload_score","workload_z"),("recovery_score","recovery_z"),("social_score","social_z")]:  # mapping activity cols to z cols
    if col in activity_clean.columns:  # true only if the activity column exists
        activity_clean = add_student_zscore(activity_clean, "student_id", col, zcol, time_col="timestamp")  # adding corresponding z-score column

sleep_clean = add_student_zscore(sleep_clean, "student_id", "sleep_hours", "sleep_z")  # adding sleep_z (no time_col passed here)

In [16]:
stress_clean.head()

,resp_time,student_id,stress_level,location,timestamp,stress_z
475,1364237696,u00,2.0,"43.70692415,-72.2873929",2013-03-25 18:54:56,0.000000
476,1364268806,u00,2.0,"43.70555193,-72.28704778",2013-03-26 03:33:26,0.000000
477,1364268814,u00,2.0,"43.70555193,-72.28704778",2013-03-26 03:33:34,0.000000
478,1364346740,u00,1.0,"43.70678675,-72.28732051",2013-03-27 01:12:20,-1.500000
480,1364437527,u00,1.0,"43.70508322,-72.28677496",2013-03-28 02:25:27,-1.095445


In [17]:
sleep_clean.head(100)

,student_id,timestamp,sleep_hours,sleep_z
1577,u00,2013-03-28 02:25:12,6.0,0.000000
1617,u00,2013-05-17 17:39:42,9.0,0.707107
1616,u00,2013-05-16 17:33:14,10.0,0.800641
1615,u00,2013-05-08 18:16:24,6.0,-0.848875
1614,u00,2013-05-06 14:40:04,9.0,0.534522
...,...,...,...,...
1501,u02,2013-04-17 18:05:16,9.0,0.645179
1489,u02,2013-03-31 17:00:24,7.0,-0.707107
1500,u02,2013-04-13 17:03:11,8.0,0.000000
1499,u02,2013-04-10 18:25:00,8.0,0.000000




---


The following code is done by Md Israk Hossain, Week 2


---





In [18]:

# ISO weeks are a standard calendar system used worldwide. They keep “week numbers” consistent (weeks starting on Monday), making weekly grouping more reliable.
stress_clean["week"] = stress_clean["timestamp"].dt.isocalendar().week.astype(int)  # taking the ISO week number from each stress timestamp and storing it as an int
activity_clean["week"] = activity_clean["timestamp"].dt.isocalendar().week.astype(int)  # adding ISO week number for each activity record
sleep_clean["week"] = sleep_clean["timestamp"].dt.isocalendar().week.astype(int)  # adding ISO week number for each sleep record


# starting to build a new dataframe containing weekly stress summaries. Here we are computing multiple summary stats for each student-week group
stress_weekly = (
    stress_clean.groupby(["student_id", "week"]).agg(
        stress_mean=("stress_level", "mean"),  # calculating average stress level during that week
        stress_std=("stress_level", "std"),  # measuring how much stress varies during that week
        stress_max=("stress_level", "max"),  # finding the highest stress level recorded in that week
        stress_count=("stress_level", "count"),  # counting how many stress readings exist in that week
        stress_z_mean=("stress_z", "mean"),  # calculating average normalized stress (z-score) in that week
        stress_z_std=("stress_z", "std"),  # measuring variation of normalized stress (z-score) in that week
        stress_z_max=("stress_z", "max"), ).reset_index() ) # finding the maximum normalized stress (z-score) in that week


agg_dict = {}  # creating a dictionary to store aggregation rules

for prefix, val_col, z_col in [  # looping through each activity score and its z-score column
    ("workload", "workload_score", "workload_z"),
    ("recovery", "recovery_score", "recovery_z"),
    ("social", "social_score", "social_z")]:

    if val_col in activity_clean.columns:  # checking if the raw score column exists
        agg_dict.update({  # adding aggregation rules for raw values
            f"{prefix}_mean": (val_col, "mean"),  # calculating weekly average
            f"{prefix}_std": (val_col, "std"),  # measuring weekly variation
            f"{prefix}_max": (val_col, "max"),  # finding weekly maximum value
            f"{prefix}_count": (val_col, "count"), }) # counting number of records in that week


    if z_col in activity_clean.columns:  # checking if normalized z-score column exists. If yes, we add aggregation rules for z-score values
        agg_dict.update({
            f"{prefix}_z_mean": (z_col, "mean"),  # calculating weekly average z-score
            f"{prefix}_z_std": (z_col, "std"),  # measuring weekly variation of z-scores
            f"{prefix}_z_max": (z_col, "max"),})  # finding highest z-score in that week

# building weekly activity summary dataframe
activity_weekly = (
    activity_clean.groupby(["student_id", "week"])  # grouping activity rows by student and week
    .agg(**agg_dict)  # applying all prepared aggregation rules
    .reset_index())   # converting grouped index back into normal columns

# building weekly sleep summary dataframe
sleep_weekly = (
    sleep_clean.groupby(["student_id", "week"]).agg(
        sleep_mean=("sleep_hours", "mean"),  # calculating weekly average sleep hours
        sleep_std=("sleep_hours", "std"),  # measuring variation of sleep hours during the week
        sleep_count=("sleep_hours", "count"),  # counting sleep records in that week
        sleep_z_mean=("sleep_z", "mean"),  # calculating weekly average normalized sleep (z-score)
        sleep_z_std=("sleep_z", "std"), ).reset_index())  # measuring variation of normalized sleep (z-score)

# building final feature table (one row per student per week)
studentlife_features = (
    stress_weekly  # using weekly stress table as the base
    .merge(activity_weekly, on=["student_id", "week"], how="left")  # attaching weekly activity features
    .merge(sleep_weekly, on=["student_id", "week"], how="left")  # attaching weekly sleep features
)

# final table containing the StudentLife multimodal weekly dataset (stress + activity + sleep together)
multimodal_df = studentlife_features.copy()  # creating a copy to avoid accidental modification of the original

In [19]:
multimodal_df.head()

,student_id,week,stress_mean,stress_std,stress_max,stress_count,stress_z_mean,stress_z_std,stress_z_max,workload_mean,...,social_max,social_count,social_z_mean,social_z_std,social_z_max,sleep_mean,sleep_std,sleep_count,sleep_z_mean,sleep_z_std
0,u00,13,2.705882,1.358524,5.0,17,0.477033,1.154053,2.035527,NaN,...,3.0,5.0,0.203169,0.926565,1.434274,7.083333,1.621354,12.0,-0.226085,0.690837
1,u00,14,2.181818,1.470930,5.0,11,-0.252880,1.063170,1.784366,NaN,...,2.0,6.0,-0.184553,0.705472,0.333333,6.666667,3.011091,6.0,-0.306360,0.950918
2,u00,15,2.166667,0.983192,3.0,6,-0.223616,0.708273,0.420137,5.400000,...,2.0,1.0,0.402200,NaN,0.402200,7.000000,3.391165,9.0,-0.173000,0.997895
3,u00,16,3.333333,0.516398,4.0,6,0.645965,0.375674,1.147061,4.333333,...,NaN,0.0,NaN,NaN,NaN,5.000000,2.738613,5.0,-0.950756,0.803546
4,u00,17,1.000000,0.000000,1.0,4,-1.146164,0.040471,-1.100449,5.200000,...,NaN,0.0,NaN,NaN,NaN,5.750000,2.500000,4.0,-0.579348,0.835402




---


The following code is done by Sahitya, Week 3


---





In [20]:
# Sahitya_week3 Created Target Variable

# Define proxy burnout risk label (multi-signal, 2-week-ahead)

print("CREATING TARGET LABEL (PROXY)")

# Two-week-ahead outcomes (ONLY for the label, never used as input features)

multimodal_df["future_stress_2w"] = multimodal_df.groupby("student_id")["stress_mean"].shift(-2)

multimodal_df["future_sleep_2w"]  = multimodal_df.groupby("student_id")["sleep_mean"].shift(-2)



# Keep only rows where we can define the 2-week-ahead outcome

multimodal_df = multimodal_df.dropna(subset=["future_stress_2w", "future_sleep_2w"]).reset_index(drop=True)



# Data-adaptive thresholds (quantile search)

# We try a small grid of quantiles and pick the pair that yields a reasonable positive rate.

# Target: ~10% to ~30% positives (enough signal, but not too many).

stress_q_candidates = [0.65, 0.70, 0.75, 0.80]

sleep_q_candidates  = [0.35, 0.30, 0.25, 0.20]



target_rate = 0.15

min_rate, max_rate = 0.10, 0.30



best = None  # (score, stress_q, sleep_q, stress_th, sleep_th, pos_rate, pos_count)



for sq in stress_q_candidates:

    stress_th = multimodal_df["future_stress_2w"].quantile(sq)

    for lq in sleep_q_candidates:

        sleep_th = multimodal_df["future_sleep_2w"].quantile(lq)



        y = ((multimodal_df["future_stress_2w"] >= stress_th) &

             (multimodal_df["future_sleep_2w"]  <= sleep_th)).astype(int)



        pos_rate = float(y.mean())

        pos_count = int(y.sum())



        # Score: prefer rates in [min_rate, max_rate] and close to target_rate

        in_range = (min_rate <= pos_rate <= max_rate)

        score = abs(pos_rate - target_rate) + (0 if in_range else 0.5)  # penalty if out of range



        if best is None or score < best[0]:

            best = (score, sq, lq, float(stress_th), float(sleep_th), pos_rate, pos_count)



_, STRESS_Q, SLEEP_Q, STRESS_TH, SLEEP_TH, POS_RATE, POS_COUNT = best



multimodal_df["burnout_risk"] = (

    (multimodal_df["future_stress_2w"] >= STRESS_TH) &

    (multimodal_df["future_sleep_2w"]  <= SLEEP_TH)

).astype(int)



print("Label created: burnout_risk")

print(f"Chosen stress quantile: {STRESS_Q}  -> STRESS_TH = {STRESS_TH:.3f}")

print(f"Chosen sleep  quantile: {SLEEP_Q}  -> SLEEP_TH  = {SLEEP_TH:.3f}")

print(f"Positive rate: {POS_RATE:.3f}  |  Positives: {POS_COUNT} / {len(multimodal_df)}")

print("Class distribution:")

print(multimodal_df["burnout_risk"].value_counts(dropna=False))



CREATING TARGET LABEL (PROXY)
Label created: burnout_risk
Chosen stress quantile: 0.65  -> STRESS_TH = 2.512
Chosen sleep  quantile: 0.35  -> SLEEP_TH  = 7.000
Positive rate: 0.164  |  Positives: 34 / 207
Class distribution:
burnout_risk
0    173
1     34
Name: count, dtype: int64




---





---


The following code is done by Ronit, Week 3


---





In [21]:
# Prepare feature sets (baseline vs multimodal)

print("="*70)

print("PREPARING FEATURE SETS")

print("="*70)



y = multimodal_df["burnout_risk"].astype(int)



# Three feature sets (Ablation Study)

# A) Stress-only

# B) Stress + Sleep

# C) Stress + Sleep + Activity (Full Multimodal)



baseline_cols = [

    # raw stress stats

    "stress_mean","stress_std","stress_max","stress_count",

    # normalized stress stats (deviation from personal baseline)

    "stress_z_mean","stress_z_std","stress_z_max",

    # temporal stress features (raw + z)

    "stress_mean_diff1","stress_mean_diff2",

    "stress_z_mean_diff1","stress_z_mean_diff2",

    "stress_mean_roll4_mean","stress_mean_roll4_std",

    "stress_z_mean_roll4_mean","stress_z_mean_roll4_std",

    # missingness flags

    "stress_missing"

]



stress_sleep_cols = baseline_cols + [

    "sleep_mean","sleep_std","sleep_count",

    "sleep_z_mean","sleep_z_std",

    "sleep_mean_diff1","sleep_mean_diff2",

    "sleep_z_mean_diff1","sleep_z_mean_diff2",

    "sleep_mean_roll4_mean","sleep_mean_roll4_std",

    "sleep_z_mean_roll4_mean","sleep_z_mean_roll4_std",

]



multimodal_cols = stress_sleep_cols + [

    # Workload (working-related)

    "workload_mean","workload_std","workload_max","workload_count",

    "workload_z_mean","workload_z_std","workload_z_max",

    "workload_mean_diff1","workload_mean_diff2",

    "workload_z_mean_diff1","workload_z_mean_diff2",

    "workload_mean_roll4_mean","workload_mean_roll4_std",

    "workload_z_mean_roll4_mean","workload_z_mean_roll4_std",



    # Recovery (relaxing-related)

    "recovery_mean","recovery_std","recovery_max","recovery_count",

    "recovery_z_mean","recovery_z_std","recovery_z_max",

    "recovery_mean_diff1","recovery_mean_diff2",

    "recovery_z_mean_diff1","recovery_z_mean_diff2",

    "recovery_mean_roll4_mean","recovery_mean_roll4_std",

    "recovery_z_mean_roll4_mean","recovery_z_mean_roll4_std",



    # Social interaction (Social2)

    "social_mean","social_std","social_max","social_count",

    "social_z_mean","social_z_std","social_z_max",

    "social_mean_diff1","social_mean_diff2",

    "social_z_mean_diff1","social_z_mean_diff2",

    "social_mean_roll4_mean","social_mean_roll4_std",

    "social_z_mean_roll4_mean","social_z_mean_roll4_std",

]



# Add missingness flags (if present)

multimodal_cols += ["workload_missing","recovery_missing","social_missing"]



# Keep only columns that exist (not all datasets have all stats)

baseline_cols     = [c for c in baseline_cols     if c in multimodal_df.columns]

stress_sleep_cols = [c for c in stress_sleep_cols if c in multimodal_df.columns]

multimodal_cols   = [c for c in multimodal_cols   if c in multimodal_df.columns]



X_baseline     = multimodal_df[baseline_cols].copy()

X_stress_sleep = multimodal_df[stress_sleep_cols].copy()

X_multimodal   = multimodal_df[multimodal_cols].copy()



groups = multimodal_df["student_id"]

weeks  = multimodal_df["week"]



print("Feature sets ready:")

print("  A) Stress-only:     ", X_baseline.shape)

print("  B) Stress+Sleep:    ", X_stress_sleep.shape)

print("  C) Full Multimodal: ", X_multimodal.shape)

PREPARING FEATURE SETS
Feature sets ready:
  A) Stress-only:      (207, 7)
  B) Stress+Sleep:     (207, 12)
  C) Full Multimodal:  (207, 33)




---


The following code is done by Md Israk Hossain, Week 3


---





In [22]:
# Defining models and parameter grids
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

print("Setting up GridSearchCV models...")

# We keep a small grid so it runs in reasonable time.
# We are using class_weight where supported because burnout-risk is often imbalanced.

models_and_params = {
    "LogReg": {
        "model": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "params": {
            "model__C": [0.1, 1.0, 10.0]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42, class_weight="balanced"),
        "params": {
            "model__n_estimators": [200, 500],
            "model__max_depth": [5, 10, None],
            "model__min_samples_split": [2, 10]
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "model__n_estimators": [100, 300],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [2, 3]
        }
    },
    "SVM": {
        "model": SVC(probability=True, class_weight="balanced"),
        "params": {
            "model__C": [0.5, 1.0, 2.0],
            "model__kernel": ["rbf"]
        }
    }
}

print("Models ready:", list(models_and_params.keys()))


Setting up GridSearchCV models...
Models ready: ['LogReg', 'RandomForest', 'GradientBoosting', 'SVM']


In [23]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
import numpy as np

def run_gridsearch(X, y, groups, model_dict, dataset_name, weeks=None):

    # Printing section header for clarity
    print(f"\n{'='*70}")
    print(f"GridSearchCV for {dataset_name} (Group-wise by student)")
    print(f"{'='*70}")

    # Creating group-based train/test split to prevent student identity leakage
    # Ensuring each student is only appearing in either train or test set
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

    # Generating indices for train and test sets
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    # Creating training and testing feature sets
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

    # Creating training and testing labels
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Storing group identifiers for train and test sets
    groups_train = groups.iloc[train_idx].reset_index(drop=True)
    groups_test  = groups.iloc[test_idx].reset_index(drop=True)

    # Separating week information if it is available
    weeks_train = weeks.iloc[train_idx].reset_index(drop=True) if weeks is not None else None
    weeks_test  = weeks.iloc[test_idx].reset_index(drop=True) if weeks is not None else None


    # Re-sampling if the test set does not contain any positive samples
    # Handling extreme class imbalance
    attempts = 0

    while y_test.sum() == 0 and attempts < 100:

        # Creating new random group split
        gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42 + attempts)

        # Generating new indices
        train_idx, test_idx = next(gss.split(X, y, groups=groups))

        # Recreating train and test sets
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Resetting group indices
        groups_train = groups.iloc[train_idx].reset_index(drop=True)
        groups_test  = groups.iloc[test_idx].reset_index(drop=True)

        # Resetting week indices if weeks are provided
        weeks_train = weeks.iloc[train_idx].reset_index(drop=True) if weeks is not None else None
        weeks_test  = weeks.iloc[test_idx].reset_index(drop=True) if weeks is not None else None

        # Increasing attempt counter
        attempts += 1


    # Printing warning if still no positive samples exist in test set
    if y_test.sum() == 0:
        print("WARNING: Despite multiple attempts, test set still contains no positive samples. ROC-AUC may be undefined.")


    # Creating dictionary to store results for each model
    results = {}

    # Initializing best model tracking variables
    best_model = None
    best_score = -1
    best_name  = None


    # Creating group-based cross-validation for training data
    # Ensuring students are not leaking across folds
    cv = GroupKFold(n_splits=3)


    # Looping through each model configuration
    for name, config in model_dict.items():

        # Printing current model name
        print(f"\nTesting {name}...")

        # Creating preprocessing + model pipeline
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),  # Filling missing values
            ("scaler",  StandardScaler()),                 # Standardizing feature values
            ("model",   config["model"])                   # Attaching ML model
        ])


        # Creating GridSearchCV for hyperparameter tuning
        grid = GridSearchCV(
            pipeline,
            config["params"],

            # Performing cross-validation while keeping groups separated
            cv=cv.split(X_train, y_train, groups=groups_train),

            # Using ROC-AUC as main evaluation metric
            scoring="roc_auc",

            # Using all CPU cores for faster training
            n_jobs=-1,

            # Suppressing verbose output
            verbose=0
        )


        # Training models for all hyperparameter combinations
        grid.fit(X_train, y_train)


        # Predicting class labels on the test set
        y_pred = grid.predict(X_test)


        # Checking if model supports probability predictions
        if hasattr(grid, "predict_proba"):

            # Getting probability of positive class
            y_pred_proba = grid.predict_proba(X_test)[:, 1]

        else:

            # Using decision scores if probability is unavailable
            scores = grid.decision_function(X_test)

            # Normalizing decision scores to probability-like values
            y_pred_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)


        # Calculating ROC-AUC if positive samples exist
        roc = roc_auc_score(y_test, y_pred_proba) if y_test.sum() > 0 else np.nan

        # Calculating Precision-Recall AUC
        pr_auc = average_precision_score(y_test, y_pred_proba) if y_test.sum() > 0 else np.nan


        # Storing all model results
        results[name] = {
            "model":       grid.best_estimator_,   # Storing best trained model
            "best_params": grid.best_params_,      # Storing best hyperparameters
            "roc_auc":     roc,
            "pr_auc":      pr_auc,
            "f1":          f1_score(y_test, y_pred, zero_division=0),
            "accuracy":    accuracy_score(y_test, y_pred),

            # Storing test data for further analysis
            "X_test":      X_test,
            "y_test":      y_test,
            "groups_test": groups_test,
            "weeks_test":  weeks_test
        }


        # Printing model performance
        print(f"ROC-AUC: {roc:.4f} | PR-AUC: {pr_auc:.4f} | F1: {results[name]['f1']:.4f} | Acc: {results[name]['accuracy']:.4f}")


        # Updating best model if ROC-AUC is higher
        if not np.isnan(roc) and roc > best_score:
            best_score = roc
            best_model = grid.best_estimator_
            best_name  = name


    # Printing best model for this dataset
    print(f"\nBest model for {dataset_name}: {best_name} (ROC-AUC={best_score:.4f})")

    # Returning all results and best model
    return results, best_model, best_name

In [24]:
print("GRIDSEARCH: BASELINE (STRESS-ONLY)")
print("="*70)

results_studentlife, best_studentlife, best_name_studentlife = run_gridsearch(
    X_baseline, y, groups, models_and_params, "Baseline", weeks=weeks
)


GRIDSEARCH: BASELINE (STRESS-ONLY)

GridSearchCV for Baseline (Group-wise by student)

Testing LogReg...
ROC-AUC: 0.5051 | PR-AUC: 0.2025 | F1: 0.2941 | Acc: 0.6066

Testing RandomForest...
ROC-AUC: 0.5604 | PR-AUC: 0.3135 | F1: 0.1538 | Acc: 0.8197

Testing GradientBoosting...
ROC-AUC: 0.5740 | PR-AUC: 0.2554 | F1: 0.2000 | Acc: 0.7377

Testing SVM...
ROC-AUC: 0.4252 | PR-AUC: 0.1732 | F1: 0.3500 | Acc: 0.5738

Best model for Baseline: GradientBoosting (ROC-AUC=0.5740)


In [25]:
print("GRIDSEARCH: STRESS + SLEEP (ABLATION)")
print("="*70)

results_stress_sleep, best_stress_sleep, best_name_stress_sleep = run_gridsearch(
    X_stress_sleep, y, groups, models_and_params, "Stress+Sleep", weeks=weeks
)


GRIDSEARCH: STRESS + SLEEP (ABLATION)

GridSearchCV for Stress+Sleep (Group-wise by student)

Testing LogReg...
ROC-AUC: 0.6429 | PR-AUC: 0.3561 | F1: 0.3333 | Acc: 0.5410

Testing RandomForest...
ROC-AUC: 0.6327 | PR-AUC: 0.3522 | F1: 0.0000 | Acc: 0.8033

Testing GradientBoosting...
ROC-AUC: 0.6054 | PR-AUC: 0.3868 | F1: 0.2667 | Acc: 0.8197

Testing SVM...
ROC-AUC: 0.6327 | PR-AUC: 0.2732 | F1: 0.3448 | Acc: 0.6885

Best model for Stress+Sleep: LogReg (ROC-AUC=0.6429)


In [26]:
print("GRIDSEARCH: MULTIMODAL (STRESS + SLEEP + ACTIVITY)")
print("="*70)

results_multimodal, best_multimodal, best_name_multimodal = run_gridsearch(
    X_multimodal, y, groups, models_and_params, "Multimodal", weeks=weeks
)


GRIDSEARCH: MULTIMODAL (STRESS + SLEEP + ACTIVITY)

GridSearchCV for Multimodal (Group-wise by student)

Testing LogReg...
ROC-AUC: 0.5595 | PR-AUC: 0.2349 | F1: 0.2632 | Acc: 0.5410

Testing RandomForest...
ROC-AUC: 0.6327 | PR-AUC: 0.3456 | F1: 0.0000 | Acc: 0.8033

Testing GradientBoosting...
ROC-AUC: 0.6939 | PR-AUC: 0.4616 | F1: 0.2500 | Acc: 0.8033

Testing SVM...
ROC-AUC: 0.5459 | PR-AUC: 0.2077 | F1: 0.0000 | Acc: 0.6393

Best model for Multimodal: GradientBoosting (ROC-AUC=0.6939)
